Implementation for traffic efficiency simulations in SUMO with detailed configuration and TraCI control:

1. Network Configuration (efficiency.net.xml)

In [ ]:
<configuration>
    <input>
        <net-file value="network.net.xml"/>
        <route-files value="vehicles.rou.xml"/>
        <additional-files value="detectors.add.xml"/>
    </input>
    <time>
        <begin value="0"/>
        <end value="3600"/> <!-- 1-hour simulation -->
        <step-length value="0.1"/> <!-- 100ms resolution -->
    </time>
    <processing>
        <time-to-teleport value="-1"/> <!-- Disable vehicle removal -->
        <collision.action value="none"/> <!-- Realistic congestion buildup -->
    </processing>
</configuration>


2. Vehicle Generation (vehicles.rou.xml)

In [ ]:
<routes>
    <vType id="car" accel="2.6" decel="4.5" sigma="0.5" length="4.3" maxSpeed="50"/>

    <flow id="main_flow" type="car" begin="0" end="3600"
          number="1200" from="edge1" to="edge3">
        <routeDistribution last="1">
            <route cost="traveltime" edges="edge1 edge2 edge3"/>
            <route cost="traveltime" edges="edge1 edge4 edge3" probability="0.3"/>
        </routeDistribution>
    </flow>

    <!-- Incident vehicle definition -->
    <vehicle id="accidentCar" depart="300">
        <route edges="edge2 edge3"/>
        <stop lane="edge2_0" endPos="150" duration="1200"/>
    </vehicle>
</routes>


3. Python Control Script (efficiency_control.py)

In [ ]:
import traci
import traci.constants as tc
import numpy as np

def calculate_alternative(veh_id):
    current_edge = traci.vehicle.getRoadID(veh_id)
    route = traci.vehicle.getRoute(veh_id)
    alt_route = traci.simulation.findRoute(
        current_edge,
        route[-1],
        routingMode=tc.ROUTING_MODE_AGGREGATED
    ).edges
    return alt_route if len(alt_route) > 0 else route

traci.start(["sumo-gui", "-c", "efficiency.net.xml"])
affected_edges = {"edge2", "edge4"}  # Congested areas

# Configure subscription for real-time monitoring
traci.vehicle.subscribeContext(
    "", tc.CMD_GET_VEHICLE_VARIABLE, 1000,
    [tc.VAR_SPEED, tc.VAR_WAITING_TIME]
)

try:
    while traci.simulation.getMinExpectedNumber() > 0:
        traci.simulationStep()

        # Dynamic rerouting logic
        for veh_id in traci.vehicle.getIDList():
            if traci.vehicle.getRoadID(veh_id) in affected_edges:
                new_route = calculate_alternative(veh_id)
                traci.vehicle.setRoute(veh_id, new_route)

        # Congestion detection and mitigation
        sub_results = traci.vehicle.getContextSubscriptionResults()
        for veh_id, data in sub_results.items():
            if data[tc.VAR_SPEED] < 2.0 and data[tc.VAR_WAITING_TIME] > 120:
                traci.vehicle.setSpeedMode(veh_id, 0)  # Disable safety checks
                traci.vehicle.slowDown(veh_id, 5, 30)  # Force gradual acceleration

except traci.FatalTraCIError:
    print("Simulation terminated early")
finally:
    traci.close()


4. Performance Metrics Analysis

In [ ]:
# Post-simulation analysis
import pandas as pd
from sumolib.output import parse

def analyze_travel_times():
    travel_data = []
    for trip in parse("tripinfo.xml", "tripinfo"):
        travel_data.append({
            "id": trip.id,
            "duration": float(trip.duration),
            "waiting": float(trip.waitingTime),
            "route": trip.route.split(),
            "depart": float(trip.depart)
        })

    df = pd.DataFrame(travel_data)
    print(f"Average travel time: {df.duration.mean():.1f}s")
    print(f"Maximum waiting time: {df.waiting.max():.1f}s")
    return df

df = analyze_travel_times()


Parameter	Value Range	Effect
accel/decel	2.0-4.5 m/s²	Controls vehicle acceleration characteristics
sigma	0.3-0.8	Driver imperfection (0=perfect, 1=random)
time-to-teleport	-1	Disables vehicle removal
step-length	0.1-1.0 s	Simulation temporal resolution
routingMode	aggregated/current	Uses historical vs real-time traffic data for routing

This implementation includes:

Real-time congestion detection using vehicle subscriptions

Adaptive routing with fallback strategies

Post-simulation performance analysis

Safety override mechanisms for gridlock situations

To optimize traffic flow, run iterative simulations with:



In [ ]:
python tools/assign/duaIterate.py -n network.net.xml -t vehicles.rou.xml -a detectors.add.xml \
--weights-priority 3 --gawron-beta 0.3 --gawron-a 0.05 --max-alternatives 5
